# WP9 simulation shard 25 of 27

Upload this standalone notebook to Colab and choose **Runtime → Run all**. Its shard index is fixed, so no cell editing is required. When the run completes, the notebook downloads its JSON result automatically.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

SHARD_INDEX = 25
NUM_SHARDS = 27
REPO_URL = os.environ.get("WDCF_REPO_URL", "https://github.com/hugogobato/wasserstein-causal-forests.git")
REPO_DIR = Path("/content/wasserstein-causal-forests")
OUTPUT_PATH = Path(f"/content/wp9_shard_{SHARD_INDEX:02d}.json")
DGPS = ("D0", "D1", "D2", "D3", "D4", "D5", "D8")
N_REGIONS = (500, 1000)
N_SEEDS = 30
N_TREES = 200
N_EVAL = 200
WORKERS = 1
CLAIM_ID = "WP9-T3-colab"

if not (REPO_DIR / "research/sim/runner.py").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-r",
    str(REPO_DIR / "requirements-colab.txt"),
], check=True)
sys.path.insert(0, str(REPO_DIR / "research"))
from sim.runner import build_simulation_tasks

tasks = build_simulation_tasks(
    dgp_names=DGPS, n_regions_list=N_REGIONS, n_seeds=N_SEEDS,
    n_trees=N_TREES, n_eval=N_EVAL, claim_id=CLAIM_ID,
    shard_index=SHARD_INDEX, num_shards=NUM_SHARDS,
)
print(f"Running shard {SHARD_INDEX:02d}/{NUM_SHARDS}: {len(tasks)} cells")
print(f"Output: {OUTPUT_PATH}")

command = [
    sys.executable, str(REPO_DIR / "research/sim/runner.py"),
    "--dgps", *DGPS,
    "--n", *(str(n) for n in N_REGIONS),
    "--seeds", str(N_SEEDS),
    "--n_trees", str(N_TREES),
    "--n_eval", str(N_EVAL),
    "--workers", str(WORKERS),
    "--shard-index", str(SHARD_INDEX),
    "--num-shards", str(NUM_SHARDS),
    "--claim", CLAIM_ID,
    "--out", str(OUTPUT_PATH),
    "--resume",
]
subprocess.run(command, cwd=REPO_DIR, check=True)

rows = json.loads(OUTPUT_PATH.read_text())
cells = {(r["dgp_id"], r["n_regions"], r["observation_regime"], r["seed"]) for r in rows}
print(f"Completed {len(cells)} cells and saved {len(rows)} rows")

try:
    from google.colab import files
    files.download(str(OUTPUT_PATH))
    print("Downloaded:", OUTPUT_PATH)
except Exception as e:
    print("(Not on Colab / download skipped):", e)
